# Przygotowanie danych do trenowania CNN dla OCR
## Dataset: Chars74K (EnglishFnt)

Ten notebook przygotowuje dane z datasetu Chars74K do trenowania sieci CNN dla zadania rozpoznawania znaków (OCR).

### 1. Import bibliotek

In [ ]:
import os
import tarfile
import urllib.request
import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms
from PIL import Image

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

### 2. Konfiguracja

In [ ]:
# URL do pobrania datasetu
DATA_URL = "http://www.ee.surrey.ac.uk/CVSSP/demos/chars74k/EnglishFnt.tgz"

# Ścieżki lokalne
DATA_DIR = "./data"
ARCHIVE_NAME = "EnglishFnt.tgz"
ARCHIVE_PATH = os.path.join(DATA_DIR, ARCHIVE_NAME)
EXTRACTED_DIR = os.path.join(DATA_DIR, "English", "Fnt")

# Parametry preprocessingu
IMAGE_SIZE = 48
MEAN = [0.485, 0.456, 0.406]  # ImageNet mean
STD = [0.229, 0.224, 0.225]   # ImageNet std

print("Konfiguracja załadowana!")

### 3. Pobieranie datasetu

In [ ]:
def download_dataset(url: str, save_path: str) -> None:
    """Pobiera dataset z podanego URL, jeśli nie istnieje lokalnie."""
    if os.path.exists(save_path):
        print(f"Plik {save_path} już istnieje. Pomijam pobieranie.")
        return
    
    print(f"Pobieranie datasetu z {url}...")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    def show_progress(block_num, block_size, total_size):
        downloaded = block_num * block_size
        percent = min(100, downloaded * 100 / total_size)
        print(f"\rPostęp: {percent:.1f}%", end="")
    
    urllib.request.urlretrieve(url, save_path, show_progress)
    print("\nPobieranie zakończone!")

# Pobierz dataset
download_dataset(DATA_URL, ARCHIVE_PATH)

### 4. Rozpakowywanie archiwum .tgz

In [ ]:
def extract_archive(archive_path: str, extract_to: str) -> None:
    """Rozpakowuje archiwum .tgz do wskazanego katalogu."""
    if os.path.exists(os.path.join(extract_to, "English", "Fnt")):
        print("Archiwum już rozpakowane. Pomijam ekstrakcję.")
        return
    
    print(f"Rozpakowywanie {archive_path}...")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(path=extract_to)
    print("Rozpakowywanie zakończone!")

# Rozpakuj archiwum
extract_archive(ARCHIVE_PATH, DATA_DIR)

### 5. Wyświetlenie przykładowego obrazu PRZED przetwarzaniem

In [ ]:
def find_sample_image(dataset_path: str) -> tuple:
    """Znajduje przykładowy obraz w datasecie."""
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                image_path = os.path.join(root, file)
                return image_path, os.path.basename(root), file
    raise FileNotFoundError("Nie znaleziono żadnych obrazów!")

# Znajdź i wczytaj przykładowy obraz
image_path, class_name, filename = find_sample_image(EXTRACTED_DIR)
sample_image = Image.open(image_path).convert('RGB')

# Wyświetl obraz PRZED przetwarzaniem
plt.figure(figsize=(8, 8))
plt.imshow(sample_image)
plt.title(f"Obraz PRZED przetwarzaniem\n"
          f"Klasa: {class_name} | Plik: {filename}\n"
          f"Rozmiar: {sample_image.size[0]}x{sample_image.size[1]} pikseli")
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Rozmiar oryginalnego obrazu: {sample_image.size}")
print(f"Tryb kolorów: {sample_image.mode}")

### 6. Definicja transformacji (preprocessing)

In [ ]:
# Transformacje dla zbioru treningowego (z data augmentation)
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),  # Zmiana rozmiaru do 48x48
    transforms.RandomHorizontalFlip(p=0.5),       # Losowe odbicie poziome
    transforms.ToTensor(),                         # Konwersja do tensora [0, 1]
    transforms.Normalize(mean=MEAN, std=STD)       # Normalizacja
])

# Transformacje dla zbioru walidacyjnego (bez augmentacji)
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

print("Transformacje treningowe:")
print(train_transform)
print("\nTransformacje walidacyjne:")
print(val_transform)

### 7. Wczytanie datasetu przy użyciu ImageFolder

In [ ]:
# Wczytaj dataset z transformacjami
train_dataset = datasets.ImageFolder(
    root=EXTRACTED_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=EXTRACTED_DIR,
    transform=val_transform
)

print(f"Liczba obrazów w datasecie: {len(train_dataset)}")
print(f"Liczba klas: {len(train_dataset.classes)}")
print(f"\nPrzykładowe klasy (foldery):")
for i, cls in enumerate(train_dataset.classes[:10]):
    print(f"  {i}: {cls}")
print(f"  ...")

### 8. Przetworzenie obrazu do tensora

In [ ]:
# Przetwórz przykładowy obraz do tensora
sample_tensor = train_transform(sample_image)

print("=" * 50)
print("TENSOR PO PRZETWORZENIU")
print("=" * 50)
print(f"Kształt tensora: {sample_tensor.shape}")
print(f"Typ danych: {sample_tensor.dtype}")
print(f"Urządzenie: {sample_tensor.device}")
print(f"Zakres wartości: [{sample_tensor.min():.4f}, {sample_tensor.max():.4f}]")
print(f"Średnia: {sample_tensor.mean():.4f}")
print(f"Odchylenie std: {sample_tensor.std():.4f}")

### 9. Dodanie wymiaru batcha

In [ ]:
# Dodaj wymiar batcha (unsqueeze na pozycji 0)
batch_tensor = sample_tensor.unsqueeze(0)

print("=" * 50)
print("TENSOR Z WYMIAREM BATCHA")
print("=" * 50)
print(f"Kształt końcowego tensora: {batch_tensor.shape}")
print(f"")
print(f"Interpretacja wymiarów:")
print(f"  - batch_tensor.shape[0] = {batch_tensor.shape[0]} (batch size)")
print(f"  - batch_tensor.shape[1] = {batch_tensor.shape[1]} (kanały RGB)")
print(f"  - batch_tensor.shape[2] = {batch_tensor.shape[2]} (wysokość)")
print(f"  - batch_tensor.shape[3] = {batch_tensor.shape[3]} (szerokość)")
print(f"")
print(f"Format: (N, C, H, W) = (batch, channels, height, width)")

### 10. Wizualizacja przetworzonego obrazu

In [ ]:
def visualize_tensor(tensor: torch.Tensor, title: str = "Przetworzony obraz") -> None:
    """Wizualizuje tensor jako obraz (po denormalizacji)."""
    # Usuń wymiar batcha jeśli istnieje
    if tensor.dim() == 4:
        tensor = tensor.squeeze(0)
    
    # Denormalizacja
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std = torch.tensor(STD).view(3, 1, 1)
    tensor_denorm = tensor * std + mean
    tensor_denorm = torch.clamp(tensor_denorm, 0, 1)
    
    # Konwersja do formatu HWC dla matplotlib
    image_np = tensor_denorm.permute(1, 2, 0).numpy()
    
    plt.figure(figsize=(6, 6))
    plt.imshow(image_np)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Wyświetl obraz PO przetworzeniu
visualize_tensor(batch_tensor, f"Obraz PO przetworzeniu ({IMAGE_SIZE}x{IMAGE_SIZE})")

### 11. Porównanie przed i po przetwarzaniu

In [ ]:
# Porównanie wizualne
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Obraz oryginalny
axes[0].imshow(sample_image)
axes[0].set_title(f"PRZED przetwarzaniem\nRozmiar: {sample_image.size}")
axes[0].axis('off')

# Obraz przetworzony (denormalizowany)
tensor_vis = batch_tensor.squeeze(0)
mean = torch.tensor(MEAN).view(3, 1, 1)
std = torch.tensor(STD).view(3, 1, 1)
tensor_denorm = torch.clamp(tensor_vis * std + mean, 0, 1)
image_processed = tensor_denorm.permute(1, 2, 0).numpy()

axes[1].imshow(image_processed)
axes[1].set_title(f"PO przetworzeniu\nRozmiar: {IMAGE_SIZE}x{IMAGE_SIZE}")
axes[1].axis('off')

plt.suptitle("Porównanie obrazu przed i po przetwarzaniu", fontsize=14)
plt.tight_layout()
plt.show()

### 12. Podsumowanie

In [ ]:
print("=" * 60)
print("PODSUMOWANIE - PRZYGOTOWANIE DANYCH DLA OCR")
print("=" * 60)
print(f"")
print(f"Dataset: Chars74K (EnglishFnt)")
print(f"Liczba obrazów: {len(train_dataset)}")
print(f"Liczba klas (znaków): {len(train_dataset.classes)}")
print(f"")
print(f"Preprocessing:")
print(f"  - Resize: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"  - Random Horizontal Flip: p=0.5")
print(f"  - ToTensor: [0, 255] -> [0.0, 1.0]")
print(f"  - Normalize: mean={MEAN}, std={STD}")
print(f"")
print(f"Kształt tensora wejściowego: {batch_tensor.shape}")
print(f"Format: (batch_size, channels, height, width)")
print(f"")
print("=" * 60)

### 13. Przykład użycia DataLoader (bonus)

In [ ]:
# Tworzenie DataLoader do trenowania
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

# Pobranie przykładowego batcha
sample_batch, sample_labels = next(iter(train_loader))

print(f"Przykładowy batch:")
print(f"  - Kształt obrazów: {sample_batch.shape}")
print(f"  - Kształt etykiet: {sample_labels.shape}")
print(f"  - Etykiety: {sample_labels[:10].tolist()}...")